1: Environment Setup & Library Imports

In [1]:
# Cell 1: Import required data engineering libraries
import os
import numpy as np
import pandas as pd

print("Data processing environment successfully initialized.")

Data processing environment successfully initialized.


2: Load Raw Datasets from Bronze Layer

In [2]:
# Cell 2: Load raw CSV files from the bronze layer directory
bronze_path = "../bronze/new/"

flow_df = pd.read_csv(f"{bronze_path}flow_schwartau.csv")
humidity_df = pd.read_csv(f"{bronze_path}humidity_schwartau.csv")
temp_df = pd.read_csv(f"{bronze_path}temperature_schwartau.csv")
weight_df = pd.read_csv(f"{bronze_path}weight_schwartau.csv")

print("Raw bronze datasets loaded successfully:")
print(f"- Flow Dataset: {flow_df.shape[0]} rows, {flow_df.shape[1]} columns")
print(f"- Humidity Dataset: {humidity_df.shape[0]} rows, {humidity_df.shape[1]} columns")
print(f"- Temperature Dataset: {temp_df.shape[0]} rows, {temp_df.shape[1]} columns")
print(f"- Weight Dataset: {weight_df.shape[0]} rows, {weight_df.shape[1]} columns")

Raw bronze datasets loaded successfully:
- Flow Dataset: 2513836 rows, 2 columns
- Humidity Dataset: 1761 rows, 2 columns
- Temperature Dataset: 253430 rows, 2 columns
- Weight Dataset: 1761 rows, 2 columns


3: Standardize Column Names

In [3]:
# Cell 3: Standardize all column names to snake_case format
def clean_columns(df):
    """Utility function to standardize column names."""
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("[^a-zA-Z0-9_]", "", regex=True)
    )
    return df

# Apply standardization across datasets
flow_df = clean_columns(flow_df)
humidity_df = clean_columns(humidity_df)
temp_df = clean_columns(temp_df)
weight_df = clean_columns(weight_df)

print("Column names successfully standardized to snake_case:")
print("Flow columns:", flow_df.columns.tolist())
print("Humidity columns:", humidity_df.columns.tolist())
print("Temperature columns:", temp_df.columns.tolist())
print("Weight columns:", weight_df.columns.tolist())

Column names successfully standardized to snake_case:
Flow columns: ['timestamp', 'flow']
Humidity columns: ['timestamp', 'humidity']
Temperature columns: ['timestamp', 'temperature']
Weight columns: ['timestamp', 'weight']


4: Handle Timestamps & UTC Standardization

In [4]:
# Cell 4: Convert timestamps to UTC datetime format and resolve flow duplicates
for df in [flow_df, humidity_df, temp_df, weight_df]:
    for col in df.columns:
        if any(key in col for key in ["time", "date", "created"]):
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

# Differentiate duplicate timestamps in flow_df (first occurrence = departure, second = arrival)
if "timestamp" in flow_df.columns:
    flow_df = flow_df.sort_values(by=["timestamp"])
    flow_df["flow_type"] = np.where(
        flow_df.duplicated(subset=["timestamp"], keep="first"),
        "arrival",
        "departure",
    )

print("Timestamps converted to UTC datetime format.")
print("Flow timestamps processed. First occurrence -> departure, second -> arrival:")
print(flow_df.head(4))

Timestamps converted to UTC datetime format.
Flow timestamps processed. First occurrence -> departure, second -> arrival:
                        timestamp  flow  flow_type
0       2017-01-01 14:15:00+00:00     0  departure
1256918 2017-01-01 14:15:00+00:00     0    arrival
1256919 2017-01-01 14:16:00+00:00     0  departure
1       2017-01-01 14:16:00+00:00     0    arrival


5: Data Cleaning & Outlier Removal

In [5]:
# Cell 5: Remove duplicates, handle missing values, and filter physical outliers
# 1. Remove exact duplicate rows across all datasets
flow_df = flow_df.drop_duplicates()
humidity_df = humidity_df.drop_duplicates()
temp_df = temp_df.drop_duplicates()
weight_df = weight_df.drop_duplicates()

# 2. Filter obvious sensor outliers (realistic ranges for weather/hive sensors)
# Temperature range: -30°C to +60°C
if "value" in temp_df.columns:
    temp_df = temp_df[(temp_df["value"] >= -30) & (temp_df["value"] <= 60)]

# Humidity range: 0% to 100%
if "value" in humidity_df.columns:
    humidity_df = humidity_df[(humidity_df["value"] >= 0) & (humidity_df["value"] <= 100)]

print("Data cleaning and outlier handling completed.")
print(f"- Cleaned Flow rows: {len(flow_df)}")
print(f"- Cleaned Humidity rows: {len(humidity_df)}")
print(f"- Cleaned Temperature rows: {len(temp_df)}")
print(f"- Cleaned Weight rows: {len(weight_df)}")

Data cleaning and outlier handling completed.
- Cleaned Flow rows: 2513601
- Cleaned Humidity rows: 1761
- Cleaned Temperature rows: 253430
- Cleaned Weight rows: 1761


6: Export Cleaned Datasets to Parquet (Silver Layer)

In [6]:
# Cell 6: Export cleaned datasets to the Silver Layer directory as Parquet files
silver_path = "../silver/"
os.makedirs(silver_path, exist_ok=True)

flow_df.to_parquet(f"{silver_path}flow_silver.parquet", index=False)
humidity_df.to_parquet(f"{silver_path}humidity_silver.parquet", index=False)
temp_df.to_parquet(f"{silver_path}temperature_silver.parquet", index=False)
weight_df.to_parquet(f"{silver_path}weight_silver.parquet", index=False)

print(f"All datasets successfully processed and saved to '{silver_path}' in Parquet format!")

All datasets successfully processed and saved to '../silver/' in Parquet format!


7: Verification (Read Parquet File)

In [7]:
# Cell 7: Verify Silver Layer Parquet dataset
test_df = pd.read_parquet(f"{silver_path}temperature_silver.parquet")
print("Verification successful. Temperature Silver Dataset sample:")
print(test_df.info())
test_df.head()

Verification successful. Temperature Silver Dataset sample:
<class 'pandas.DataFrame'>
RangeIndex: 253430 entries, 0 to 253429
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype              
---  ------       --------------   -----              
 0   timestamp    253430 non-null  datetime64[us, UTC]
 1   temperature  251398 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(1)
memory usage: 3.9 MB
None


,timestamp,temperature
0,2017-01-01 14:10:00+00:00,NaN
1,2017-01-01 14:15:00+00:00,12.34
2,2017-01-01 14:20:00+00:00,12.27
3,2017-01-01 14:25:00+00:00,12.28
4,2017-01-01 14:30:00+00:00,12.36
